# Step 3 — Vectorization & FAISS Index

This notebook reads `chunks/chunks.csv` (output of `rag_step2_chunking.ipynb`), encodes every chunk
into a 768-dim vector using each of the three fine-tuned encoders, and saves one FAISS index per model.

**Prerequisite:** `baseline_step1.ipynb` must have been run with weight saving enabled, producing:
```
weights/bert-base-uncased_IHC/   or   weights/bert-base-uncased_ISHate/
weights/hateBERT_IHC/            or   weights/hateBERT_ISHate/
weights/roberta-base_IHC/        or   weights/roberta-base_ISHate/
```

**Outputs:**
```
index/bert-base-uncased.faiss
index/hateBERT.faiss
index/roberta-base.faiss
index/documents.json          ← shared lookup table: index position → original text string
```

**Key design choice — encoder loading:**  
Fine-tuned weights include a classification head (`classifier.weight/bias`). For embedding we only
need the base transformer encoder. We load with `AutoModel` (not `AutoModelForSequenceClassification`):
HuggingFace will load the base weights and emit a harmless warning about the ignored classifier head.
The `[CLS]` token output (`last_hidden_state[:, 0, :]`) is the 768-dim chunk embedding.

**Similarity metric:** cosine similarity, implemented as inner product after L2-normalizing vectors
(`faiss.normalize_L2`). Index type: `IndexFlatIP` (exact search, no approximation).

## 1. Imports

In [ ]:
# TODO: imports
# - pandas, numpy
# - torch
# - faiss
# - json, os
# - transformers: AutoTokenizer, AutoModel
# - tqdm (for progress bar during encoding)

## 2. Configuration

Edit `DATASET_USED` to switch between fine-tuned checkpoints (IHC vs ISHate).
The HuggingFace model ID (`hf_id`) is used as fallback if the local weights folder does not exist yet.

In [ ]:
# TODO: define config dict — example structure:
#
# DATASET_USED = "ISHate"   # or "IHC" — determines which fine-tuned checkpoint to load
#
# MODEL_CONFIGS = {
#     "bert-base-uncased": {
#         "weights_path": f"weights/bert-base-uncased_{DATASET_USED}",
#         "hf_fallback":  "bert-base-uncased",
#     },
#     "hateBERT": {
#         "weights_path": f"weights/hateBERT_{DATASET_USED}",
#         "hf_fallback":  "GroNLP/hateBERT",
#     },
#     "roberta-base": {
#         "weights_path": f"weights/roberta-base_{DATASET_USED}",
#         "hf_fallback":  "roberta-base",
#     },
# }
#
# BATCH_SIZE = 64
# MAX_LENGTH = 128    # max tokens per chunk — must match chunking notebook
# EMBED_DIM  = 768    # [CLS] vector dimension for all three models

## 3. Load Chunks

Load `chunks/chunks.csv`. The `text` column contains the string that will be encoded.
The `chunk_id` column is the row index and will match the FAISS vector position.

In [ ]:
# TODO:
# df = pd.read_csv("chunks/chunks.csv")
# texts = df["text"].tolist()
# print(f"Total chunks to index: {len(texts):,}")
# print(df["source"].value_counts())

## 4. Encoding Function

Takes a list of strings + a loaded model + tokenizer, returns a float32 numpy array of shape
`(N, 768)` where each row is the `[CLS]` embedding of one chunk.

Uses batched inference with `torch.no_grad()`. Moves tensors to GPU if available.

In [ ]:
# TODO: define encode(texts, model, tokenizer, batch_size, max_length) function
#
# For each batch:
#   1. tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=max_length)
#   2. move input tensors to model.device
#   3. with torch.no_grad(): outputs = model(**inputs)
#   4. cls_vec = outputs.last_hidden_state[:, 0, :].cpu().numpy()  ← [CLS] token
#   5. append to list
#
# return np.vstack(all_vecs).astype("float32")

## 5. Build One FAISS Index Per Model

For each model in `MODEL_CONFIGS`:
1. Determine which path to load from: local `weights_path` if the folder exists, else `hf_fallback`
2. Load tokenizer and model (`AutoModel`, not classification)
3. Encode all chunks (show progress with tqdm)
4. L2-normalize vectors (`faiss.normalize_L2`) for cosine similarity
5. Build `IndexFlatIP(768)` and add all vectors
6. Save index to `index/{model_name}.faiss`
7. Print: total vectors added, index size on disk

In [ ]:
# TODO: loop over MODEL_CONFIGS
#
# for model_name, cfg in MODEL_CONFIGS.items():
#   load_path = cfg["weights_path"] if os.path.isdir(cfg["weights_path"]) else cfg["hf_fallback"]
#   print(f"Loading {model_name} from: {load_path}")
#
#   tokenizer = AutoTokenizer.from_pretrained(load_path)
#   model     = AutoModel.from_pretrained(load_path)
#   model.eval()
#   # move to GPU if available: model.to(device)
#
#   vectors = encode(texts, model, tokenizer, BATCH_SIZE, MAX_LENGTH)
#   faiss.normalize_L2(vectors)
#
#   index = faiss.IndexFlatIP(EMBED_DIM)
#   index.add(vectors)
#
#   os.makedirs("index", exist_ok=True)
#   faiss.write_index(index, f"index/{model_name}.faiss")
#   print(f"  Saved index/{model_name}.faiss  ({index.ntotal} vectors)")
#
#   del model  ← free GPU memory before loading next model

## 6. Save Shared Lookup Table

FAISS returns integer positions, not text. We save `documents.json` as a list of strings
so that position `i` in the FAISS index maps to `documents[i]` in the JSON.

This lookup table is **shared across all three indexes** — the chunk order is identical.

In [ ]:
# TODO:
# with open("index/documents.json", "w") as f:
#     json.dump(texts, f, ensure_ascii=False, indent=2)
# print(f"Saved index/documents.json  ({len(texts):,} entries)")

## 7. Sanity Check

Load the hateBERT index and query it with a sample tweet to verify the pipeline end-to-end.
We should see similar hate / non-hate examples retrieved with high similarity scores.

Also test with a clearly non-hateful tweet to confirm retrieved chunks are relevant.

In [ ]:
# TODO:
# 1. load index/hateBERT.faiss and index/documents.json
#
# 2. define query_index(tweet_text, index, tokenizer, model, k=5, threshold=0.7):
#    - encode the single tweet → vector shape (1, 768)
#    - faiss.normalize_L2(vec)
#    - scores, positions = index.search(vec, k)
#    - filter: keep only positions where scores[0][i] >= threshold
#    - return [(documents[pos], score) for pos, score in zip(positions[0], scores[0]) if score >= threshold]
#
# 3. test queries:
#    query_index("I hate immigrants, they should all leave")
#    query_index("Let's celebrate diversity in our community")
#
# 4. print retrieved chunks + similarity scores for each query
# Expected: hate tweet retrieves hate-labelled chunks; benign tweet retrieves non-hate or unknown